# Induced two-color subgraph with native quantum functions

Select the largest induced subgraph that admits two colors. Nodes may remain
uncolored; no node has two colors and no edge joins same-color selected nodes.
Keep the original six-node graph, two colors, eight layers and fifty optimizer
iterations. The full local search can take about forty minutes on the measured CPU.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

import flagquantum as fq
from examples.synthesis.applications.optimization.max_induced_k_color_subgraph.overview.plots import (
    optimization_figure,
)
from examples.synthesis.applications.optimization.max_induced_k_color_subgraph.overview.workflow import (
    EDGES,
    build_program,
    model_row,
    run_workflow,
)

graph = nx.erdos_renyi_graph(6, 0.5, seed=7)
assert tuple(graph.edges()) == EDGES
print("Original model:", "two colors; no double color; no same-color edge")
print("Objective:", "maximize the number of colored nodes")


## Native quantum program and explicit synthesis limits

The adjacent `main.py` defines `@qfunc main(params, a, b)`: allocate six bits per
color, prepare uniform superpositions, then apply eight cost and RX mixer layers.
Reuse that native function. Read the full named `(a, b)` distribution. Numeric bit
i denotes node i. The 16 MiB state-planning budget is not total process RSS.


In [ ]:
initial_program = build_program()
initial_result = fq.run(initial_program)
initial_readout = dict(
    initial_program.probabilities(initial_result, registers=("a", "b"))
)
print("Joint assignments:", len(initial_readout))
print("Resources:", initial_program.report["resources"])
print("Temporary workspace:", initial_program.report["workspace_qubits"])


## Optimize using actual native execution

Every objective evaluation synthesizes and executes the twelve-qubit program.
Native Nelder-Mead uses the declared sixteen-angle start and simplex step 0.07;
exact local CVaR replaces private shot-based objective estimation. Retain the
50-iteration budget and quantile 0.7. No identical optimizer trajectory or
convergence is promised. This cell performs the complete original-size search.


In [ ]:
workflow = run_workflow(maxiter=50, shots=1000, seed=2718)
print("Initial CVaR:", workflow.trace[0].cost)
print("Final CVaR:", workflow.optimizer.value)
print(
    "Iterations / evaluations:",
    workflow.optimizer.iterations,
    workflow.optimizer.evaluations,
)
print("Converged:", workflow.optimizer.converged)


## Actual observations and a separate classical reference

Draw 1000 optimized samples and 1000 seeded uniform samples. Rank only observed
feasible rows by colored-node count. The retained quantum penalty is distinct
from original feasibility and objective. If no feasible coloring is sampled,
report `None`. Enumerate the bounded classical reference only after sampling;
it never supplies parameters, quantum probabilities or observed answers.


In [ ]:
print("Best actually observed feasible coloring:", workflow.best_feasible)
for name, samples in (("optimized", workflow.samples), ("uniform", workflow.uniform)):
    print(name, "observations:", sum(row.count for row in samples))
    print(
        name,
        "feasible observations:",
        sum(row.count for row in samples if row.feasible),
    )
print("Separate exact bounded reference:", model_row(workflow.exact_reference))
print("No convergence, sampled optimum or quantum advantage claim is made.")


## Source graph, evaluation trace, cost histogram and observed coloring

Both empirical histograms use the same fifty bins. Blue and coral distinguish
the two colors; gray nodes are unselected. If no feasible coloring was observed,
the figure states that instead of drawing the classical reference as a sample.


In [ ]:
figure = optimization_figure(workflow)
plt.show()
